# VisoLabel Colab Trainer

Use this notebook when VisoLabel gives you a Colab training token.

1. In VisoLabel, choose `Train on Colab` and copy the token.
2. In Colab, set `Runtime > Change runtime type > GPU`.
3. Run the form cell below.
4. Paste the token and click `Run full pipeline`.

The notebook downloads the encrypted bundle, decrypts it locally in Colab, validates the dataset, trains the selected model, and uploads checkpoints back to VisoLabel.


In [ ]:

#@title Launch VisoLabel Training GUI { display-mode: "form" }
#@markdown Run this cell once. The GUI handles token resolution, bundle download/decryption, training, and result upload.
import base64
import hashlib
import html
import io
import json
import math
import os
import shutil
import subprocess
import sys
import textwrap
import time
import zipfile
from pathlib import Path

os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def _pip_install_once():
    import importlib.util
    required = {
        "requests": "requests",
        "cryptography": "cryptography",
        "ipywidgets": "ipywidgets",
        "tqdm": "tqdm",
    }
    missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

_pip_install_once()

import requests
from IPython.display import HTML, display
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from tqdm.auto import tqdm

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

import ipywidgets as widgets

DEFAULT_API_BASE_URL = os.environ.get("VISOLABEL_API_BASE", "https://api.visolabel.ovh")
TOKEN_PREFIX = "VL1."
LEGACY_TOKEN_PREFIX = "VL-COLAB-ENC1."
WORK_DIR = Path("/content/visolabel_run")
BUNDLE_ENCRYPTED = WORK_DIR / "bundle.zip.vlenc"
BUNDLE_ZIP = WORK_DIR / "bundle.zip"
BUNDLE_DIR = WORK_DIR / "bundle"
OUTPUT_DIR = WORK_DIR / "output"
RUNNER_PATH = WORK_DIR / "visolabel_train_runner.py"
TRAINING_LOG = OUTPUT_DIR / "training.log"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
ARTIFACT_EXTENSIONS = {".pth", ".pt", ".ckpt", ".onnx", ".log", ".json", ".yaml", ".yml", ".txt", ".csv"}


RUNNER_CODE = r"""
import json
import math
import os
import shutil
import sys
import time
from pathlib import Path

os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def log(message):
    print(f"[visolabel-train] {message}", flush=True)


def import_attr(module, names):
    for name in names:
        if hasattr(module, name):
            return getattr(module, name), name
    raise RuntimeError("None of these RF-DETR classes are available: " + ", ".join(names))


def model_class_for_name(model_name):
    import rfdetr
    name = str(model_name or "RF-DETR-N")
    upper = name.upper()
    is_seg = "SEG" in upper
    suffix = upper.split("-")[-1]
    if suffix == "2XL":
        det = ["RFDETR2XLarge", "RFDETRXLarge", "RFDETRLarge"]
        seg = ["RFDETRSeg2XLarge", "RFDETRSegXLarge", "RFDETRSegLarge"]
    elif suffix in {"XL", "X"}:
        det = ["RFDETRXLarge", "RFDETRLarge"]
        seg = ["RFDETRSegXLarge", "RFDETRSegLarge"]
    elif suffix in {"L", "LARGE"}:
        det = ["RFDETRLarge", "RFDETRBase"]
        seg = ["RFDETRSegLarge", "RFDETRSegMedium", "RFDETRSegSmall", "RFDETRSegNano"]
    elif suffix in {"M", "MEDIUM"}:
        det = ["RFDETRMedium", "RFDETRBase"]
        seg = ["RFDETRSegMedium", "RFDETRSegSmall", "RFDETRSegNano"]
    elif suffix in {"S", "SMALL"}:
        det = ["RFDETRSmall", "RFDETRBase"]
        seg = ["RFDETRSegSmall", "RFDETRSegNano"]
    else:
        det = ["RFDETRNano", "RFDETRBase"]
        seg = ["RFDETRSegNano", "RFDETRSegSmall"]
    return import_attr(rfdetr, seg if is_seg else det)


def nearest_valid_resolution(requested, block_size):
    requested = int(requested or 560)
    block_size = max(int(block_size or 1), 1)
    if requested % block_size == 0:
        return requested
    lower = max(block_size, (requested // block_size) * block_size)
    upper = lower + block_size
    return lower if requested - lower <= upper - requested else upper


def apply_resolution(model, requested):
    config = getattr(model, "model_config", None)
    if config is None:
        return int(requested or 560)
    patch_size = int(getattr(config, "patch_size", 14) or 14)
    num_windows = int(getattr(config, "num_windows", 1) or 1)
    block_size = patch_size * num_windows
    resolution = nearest_valid_resolution(requested, block_size)
    current_resolution = int(getattr(config, "resolution", resolution) or resolution)
    current_pe = getattr(config, "positional_encoding_size", None)
    if current_pe is not None:
        derived_pe = current_resolution // patch_size
        if int(current_pe) == int(derived_pe):
            setattr(config, "positional_encoding_size", resolution // patch_size)
    setattr(config, "resolution", resolution)
    wrapped = getattr(model, "model", None)
    if wrapped is not None:
        if hasattr(wrapped, "resolution"):
            wrapped.resolution = resolution
        args = getattr(wrapped, "args", None)
        if args is not None:
            if hasattr(args, "resolution"):
                args.resolution = resolution
            if hasattr(args, "positional_encoding_size") and current_pe is not None and int(current_pe) == int(current_resolution // patch_size):
                args.positional_encoding_size = resolution // patch_size
    if resolution != int(requested or resolution):
        log(f"Adjusted RF-DETR resolution from {requested} to {resolution}; model requires multiples of {block_size}.")
    return resolution


def make_train_kwargs(cfg, dataset_dir, output_dir, resolution):
    batch_size = int(cfg.get("batch_size", 4) or 4)
    grad_accum_steps = int(cfg.get("grad_accum_steps") or max(1, math.ceil(16 / max(batch_size, 1))))
    kwargs = {
        "dataset_dir": str(dataset_dir),
        "output_dir": str(output_dir),
        "epochs": int(cfg.get("epochs", 50) or 50),
        "batch_size": batch_size,
        "grad_accum_steps": grad_accum_steps,
        "lr": float(cfg.get("learning_rate", cfg.get("lr", 1e-4)) or 1e-4),
        "resolution": int(resolution),
        "class_names": cfg.get("classes") or None,
        "device": "cuda",
        "num_workers": int(cfg.get("num_workers", 2) or 2),
        "progress_bar": True,
        "tensorboard": False,
        "wandb": False,
        "mlflow": False,
        "clearml": False,
        "checkpoint_interval": int(cfg.get("checkpoint_interval", 1) or 1),
        "eval_interval": int(cfg.get("eval_interval", 1) or 1),
        "eval_max_dets": int(cfg.get("eval_max_dets", 500) or 500),
        "log_per_class_metrics": True,
        "compute_val_loss": False,
        "run_test": False,
    }
    return {k: v for k, v in kwargs.items() if v is not None}


def build_train_config(model, kwargs):
    attempt = dict(kwargs)
    attempt.pop("resolution", None)
    attempt.pop("device", None)
    optional_order = [
        "run_test", "compute_val_loss", "log_per_class_metrics", "eval_interval",
        "eval_max_dets", "clearml", "mlflow", "class_names", "num_workers",
    ]
    while True:
        try:
            return model.get_train_config(**attempt), attempt
        except Exception as exc:
            removed = None
            for key in optional_order:
                if key in attempt:
                    removed = key
                    attempt.pop(key, None)
                    log(f"RF-DETR TrainConfig rejected '{key}', retrying without it.")
                    break
            if removed is None:
                raise exc


def train_with_custom_lightning(model, kwargs):
    from rfdetr.training import RFDETRDataModule, RFDETRModelModule, build_trainer
    train_config, used_kwargs = build_train_config(model, kwargs)
    dataset_dir = getattr(train_config, "dataset_dir", None)
    if dataset_dir and hasattr(model, "_align_num_classes_from_dataset"):
        model._align_num_classes_from_dataset(dataset_dir)
    module = RFDETRModelModule(model.model_config, train_config)
    datamodule = RFDETRDataModule(model.model_config, train_config)
    trainer = build_trainer(
        train_config,
        model.model_config,
        accelerator="gpu" if os.environ.get("CUDA_VISIBLE_DEVICES", "") != "-1" else "auto",
        devices=1,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        enable_progress_bar=True,
    )
    log("Starting RF-DETR Lightning fit with sanity validation disabled.")
    trainer.fit(module, datamodule, ckpt_path=getattr(train_config, "resume", None) or None)
    if hasattr(model, "model") and hasattr(model.model, "model"):
        model.model.model = module.model
    return used_kwargs


def train_with_high_level(model, kwargs):
    log("Starting RF-DETR high-level train() fallback.")
    return model.train(**kwargs)


def run_detection(manifest_path, dataset_dir, output_dir):
    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
    cfg = manifest.get("training_config", {})
    model_name = cfg.get("model_name", "RF-DETR-N")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cls, selected_name = model_class_for_name(model_name)
    log(f"Requested model {model_name}; using {selected_name}.")
    model = cls()
    resolution = apply_resolution(model, cfg.get("image_size", cfg.get("resolution", 560)))
    kwargs = make_train_kwargs(cfg, dataset_dir, output_dir, resolution)
    try:
        train_with_custom_lightning(model, kwargs)
    except Exception as exc:
        log(f"Custom Lightning path failed: {exc}")
        train_with_high_level(model, kwargs)
    log("RF-DETR training finished.")


def run_classifier(manifest_path, dataset_dir, output_dir):
    import torch
    from torch import nn
    from torch.utils.data import DataLoader
    from torchvision import datasets, transforms
    import timm

    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
    cfg = manifest.get("training_config", {})
    image_size = int(cfg.get("image_size", 224) or 224)
    batch_size = int(cfg.get("batch_size", 16) or 16)
    epochs = int(cfg.get("epochs", 30) or 30)
    lr = float(cfg.get("learning_rate", 3e-4) or 3e-4)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    train_dir = Path(dataset_dir) / "train"
    valid_dir = Path(dataset_dir) / "valid"
    if not valid_dir.exists():
        valid_dir = Path(dataset_dir) / "val"
    if not train_dir.exists():
        raise RuntimeError(f"Classification train directory missing: {train_dir}")
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(image_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_ds = datasets.ImageFolder(str(train_dir), transform=train_tf)
    val_ds = datasets.ImageFolder(str(valid_dir), transform=val_tf) if valid_dir.exists() else None
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True) if val_ds else None
    model_name = str(cfg.get("model_name", "convnext_tiny.fb_in22k_ft_in1k"))
    model = timm.create_model(model_name, pretrained=True, num_classes=len(train_ds.classes))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=float(cfg.get("label_smoothing", 0.0) or 0.0))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=float(cfg.get("weight_decay", 0.05) or 0.05))
    best_path = output_dir / "checkpoint_best_total.pth"
    best_acc = -1.0
    log(f"Starting {model_name} classifier training on {device}.")
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item())
        if val_loader:
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(device), y.to(device)
                    pred = model(x).argmax(dim=1)
                    correct += int((pred == y).sum().item())
                    total += int(y.numel())
            acc = 100.0 * correct / max(total, 1)
            log(f"Epoch {epoch + 1}/{epochs}: loss={total_loss / max(len(train_loader), 1):.4f}, val_acc={acc:.2f}%")
            if acc >= best_acc:
                best_acc = acc
                torch.save({"model_state_dict": model.state_dict(), "classes": train_ds.classes, "model_name": model_name}, best_path)
        else:
            log(f"Epoch {epoch + 1}/{epochs}: loss={total_loss / max(len(train_loader), 1):.4f}")
    if not best_path.exists():
        torch.save({"model_state_dict": model.state_dict(), "classes": train_ds.classes, "model_name": model_name}, best_path)
    log("Classifier training finished.")


def main():
    manifest_path = Path(sys.argv[1])
    dataset_dir = Path(sys.argv[2])
    output_dir = Path(sys.argv[3])
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    command = manifest.get("task", {}).get("command", "train")
    if command == "classify":
        run_classifier(manifest_path, dataset_dir, output_dir)
    elif command == "train":
        run_detection(manifest_path, dataset_dir, output_dir)
    else:
        raise RuntimeError(f"Unsupported training command: {command}")


if __name__ == "__main__":
    main()
"""


def _b64url_decode(value: str) -> bytes:
    value = value.strip()
    return base64.urlsafe_b64decode(value + "=" * (-len(value) % 4))


def _normalize_url(base: str, endpoint: str) -> str:
    return base.rstrip("/") + "/" + endpoint.lstrip("/")


def _format_bytes(size: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(size)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.1f} {unit}" if unit != "B" else f"{int(value)} B"
        value /= 1024


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def _count_images(path: Path) -> int:
    if not path or not path.exists():
        return 0
    return sum(1 for p in path.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def _safe_extract_zip(zip_path: Path, target: Path):
    target_resolved = target.resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            destination = (target / member.filename).resolve()
            if not str(destination).startswith(str(target_resolved)):
                raise RuntimeError(f"Unsafe zip path: {member.filename}")
        zf.extractall(target)


def _normalize_encryption_metadata(metadata: dict) -> dict:
    if not isinstance(metadata, dict):
        return {}
    if metadata.get("salt_b64") and metadata.get("nonce_b64") and metadata.get("tag_b64"):
        normalized = dict(metadata)
    else:
        normalized = {
            "enabled": True,
            "algorithm": metadata.get("algorithm", "AES-256-GCM"),
            "kdf": metadata.get("kdf", "PBKDF2-HMAC-SHA256"),
            "kdf_iterations": metadata.get("kdf_iterations", metadata.get("i", 200000)),
            "salt_b64": metadata.get("s") or metadata.get("salt_b64") or metadata.get("salt", ""),
            "nonce_b64": metadata.get("n") or metadata.get("nonce_b64") or metadata.get("nonce", ""),
            "tag_b64": metadata.get("g") or metadata.get("tag_b64") or metadata.get("tag") or metadata.get("t", ""),
            "aad_b64": metadata.get("a") or metadata.get("aad_b64") or metadata.get("aad", ""),
            "plaintext_sha256": metadata.get("ps") or metadata.get("plaintext_sha256") or metadata.get("psha256", ""),
            "ciphertext_sha256": metadata.get("cs") or metadata.get("ciphertext_sha256") or metadata.get("csha256", ""),
            "plaintext_size": metadata.get("psz") or metadata.get("plaintext_size", 0),
            "ciphertext_size": metadata.get("csz") or metadata.get("ciphertext_size", 0),
        }
    normalized.setdefault("algorithm", "AES-256-GCM")
    normalized.setdefault("kdf", "PBKDF2-HMAC-SHA256")
    normalized.setdefault("kdf_iterations", normalized.get("iterations", 200000))
    normalized.setdefault("aad_b64", "dmlzb2xhYmVsLWNvbGFiLWJ1bmRsZS12MQ==")
    if not normalized.get("salt_b64") or not normalized.get("nonce_b64") or not normalized.get("tag_b64"):
        return {}
    return normalized


def _extract_bundle_url(value: dict) -> str:
    if not isinstance(value, dict):
        return ""
    for key in ("bundle_url", "download_url", "presigned_download_url", "s3_url", "url"):
        found = str(value.get(key) or "")
        if found:
            return found
    nested = value.get("bundle") or value.get("data") or {}
    if isinstance(nested, dict):
        return _extract_bundle_url(nested)
    return ""


def _extract_encryption_metadata(value: dict) -> dict:
    if not isinstance(value, dict):
        return {}
    direct = _normalize_encryption_metadata(value.get("encryption") or value.get("encryption_metadata") or value.get("e") or {})
    if direct:
        return direct
    nested = value.get("bundle") or value.get("data") or {}
    if isinstance(nested, dict):
        return _extract_encryption_metadata(nested)
    return {}


def _parse_encrypted_token(token: str):
    prefix = TOKEN_PREFIX if token.startswith(TOKEN_PREFIX) else LEGACY_TOKEN_PREFIX
    payload = json.loads(_b64url_decode(token[len(prefix):]).decode("utf-8"))
    server_token = ""
    password = ""
    api_url = ""
    encryption = {}
    bundle_url = ""
    bundle_id = ""
    if isinstance(payload, list):
        if len(payload) >= 1:
            server_token = str(payload[0] or "")
        if len(payload) >= 2:
            password = str(payload[1] or "")
        if len(payload) >= 3:
            if isinstance(payload[2], dict):
                encryption = _normalize_encryption_metadata(payload[2])
            else:
                api_url = str(payload[2] or "")
        if len(payload) >= 4 and isinstance(payload[3], dict):
            encryption = _normalize_encryption_metadata(payload[3])
    elif isinstance(payload, dict):
        server_token = str(payload.get("server_token") or payload.get("token") or "")
        password = str(payload.get("password") or "")
        api_url = str(payload.get("api_url") or payload.get("base_url") or "")
        bundle_url = _extract_bundle_url(payload)
        encryption = _extract_encryption_metadata(payload)
        bundle_id = str(payload.get("bundle_id") or "")
    else:
        raise RuntimeError("Encrypted token payload is not valid JSON.")
    return {
        "server_token": server_token,
        "password": password,
        "api_url": api_url,
        "bundle_url": bundle_url,
        "encryption": encryption,
        "bundle_id": bundle_id,
    }


def _validate_coco_split(split_dir: Path):
    ann_path = split_dir / "_annotations.coco.json"
    result = {"split": split_dir.name, "images": 0, "annotations": 0, "categories": 0, "removed_annotations": 0}
    if not ann_path.exists():
        if split_dir.name == "valid":
            alt = split_dir.parent / "val" / "_annotations.coco.json"
            if alt.exists():
                shutil.copytree(alt.parent, split_dir, dirs_exist_ok=True)
            else:
                raise RuntimeError(f"Missing COCO annotations: {ann_path}")
        else:
            raise RuntimeError(f"Missing COCO annotations: {ann_path}")
    data = json.loads(ann_path.read_text(encoding="utf-8"))
    images = data.get("images") or []
    anns = data.get("annotations") or []
    categories = data.get("categories") or []
    image_ids = {img.get("id") for img in images}
    category_ids = {cat.get("id") for cat in categories}
    valid_anns = []
    for ann in anns:
        bbox = ann.get("bbox") or []
        if ann.get("image_id") not in image_ids or ann.get("category_id") not in category_ids:
            result["removed_annotations"] += 1
            continue
        if len(bbox) != 4 or float(bbox[2]) <= 0 or float(bbox[3]) <= 0:
            result["removed_annotations"] += 1
            continue
        ann.setdefault("area", float(bbox[2]) * float(bbox[3]))
        ann.setdefault("iscrowd", 0)
        valid_anns.append(ann)
    existing = {p.name for p in split_dir.iterdir() if p.is_file()}
    missing_images = [img.get("file_name") for img in images if img.get("file_name") not in existing]
    if missing_images:
        raise RuntimeError(f"{split_dir.name} annotations reference missing image files. First missing: {missing_images[0]}")
    data["annotations"] = valid_anns
    ann_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    result.update({"images": len(images), "annotations": len(valid_anns), "categories": len(categories)})
    return result


def _validate_detection_dataset(dataset_dir: Path):
    reports = []
    train_dir = dataset_dir / "train"
    valid_dir = dataset_dir / "valid"
    if not valid_dir.exists() and (dataset_dir / "val").exists():
        shutil.copytree(dataset_dir / "val", valid_dir, dirs_exist_ok=True)
    for split in (train_dir, valid_dir):
        reports.append(_validate_coco_split(split))
    if reports[0]["images"] <= 0:
        raise RuntimeError("Training split has no images.")
    if reports[1]["images"] <= 0:
        raise RuntimeError("Validation split has no images. RF-DETR requires a non-empty validation split.")
    return reports


def _validate_classification_dataset(dataset_dir: Path):
    train = dataset_dir / "train"
    if not train.exists():
        raise RuntimeError(f"Missing classification train directory: {train}")
    classes = [p for p in train.iterdir() if p.is_dir()]
    if not classes:
        raise RuntimeError("Classification train directory has no class folders.")
    return [{"split": "train", "images": _count_images(train), "categories": len(classes), "annotations": 0, "removed_annotations": 0}]


class VisoLabelColabGUI:
    def __init__(self):
        self.session = requests.Session()
        self.token_value = ""
        self.api_base = DEFAULT_API_BASE_URL
        self.api_token = ""
        self.manifest = None
        self.cfg = None
        self.task = ""
        self.command = ""
        self.dataset_dir = None
        self.training_finished = False
        self.started_at = None
        self.upload_list = []

        self.token = widgets.Password(
            description="Token",
            placeholder="Paste VisoLabel Colab token",
            layout=widgets.Layout(width="100%"),
            style={"description_width": "70px"},
        )
        self.api_url = widgets.Text(
            value=DEFAULT_API_BASE_URL,
            description="API URL",
            layout=widgets.Layout(width="100%"),
            style={"description_width": "70px"},
        )
        self.run_button = widgets.Button(description="Run full pipeline", button_style="primary", layout=widgets.Layout(width="170px"))
        self.resolve_button = widgets.Button(description="Resolve bundle", layout=widgets.Layout(width="145px"))
        self.train_button = widgets.Button(description="Start training", layout=widgets.Layout(width="145px"), disabled=True)
        self.upload_button = widgets.Button(description="Upload results", layout=widgets.Layout(width="145px"), disabled=True)
        self.clear_button = widgets.Button(description="Clear logs", layout=widgets.Layout(width="115px"))
        self.status = widgets.HTML()
        self.summary = widgets.HTML()
        self.progress = widgets.FloatProgress(value=0, min=0, max=100, description="Progress", bar_style="")
        self.log_output = widgets.Output(layout=widgets.Layout(border="1px solid #d0d7de", max_height="460px", overflow_y="auto", padding="8px"))

        self.run_button.on_click(self._on_run_full)
        self.resolve_button.on_click(self._on_resolve)
        self.train_button.on_click(self._on_train)
        self.upload_button.on_click(self._on_upload)
        self.clear_button.on_click(lambda _: self.log_output.clear_output())
        self._set_status("Ready", "Paste the VisoLabel token and run the pipeline.", "")
        self._render_summary()

    def display(self):
        display(HTML("""
        <style>
          .vl-hero {border:1px solid #d0d7de; border-radius:8px; padding:14px 16px; margin-bottom:10px; background:#f6f8fa;}
          .vl-hero h2 {font-size:20px; margin:0 0 4px 0;}
          .vl-hero p {margin:0; color:#57606a;}
          .vl-cards {display:grid; grid-template-columns:repeat(auto-fit,minmax(150px,1fr)); gap:8px; margin:10px 0;}
          .vl-card {border:1px solid #d0d7de; border-radius:8px; padding:10px; background:white;}
          .vl-label {font-size:12px; color:#57606a;}
          .vl-value {font-weight:600; margin-top:2px; overflow-wrap:anywhere;}
          .vl-sub {font-size:12px; color:#57606a; margin-top:2px; overflow-wrap:anywhere;}
          .vl-status {border:1px solid #d0d7de; border-radius:8px; padding:10px; margin:10px 0;}
          .vl-status.ok {border-color:#2da44e; background:#dafbe1;}
          .vl-status.err {border-color:#cf222e; background:#ffebe9;}
          .vl-status.active {border-color:#0969da; background:#ddf4ff;}
          .vl-status h3 {font-size:15px; margin:0 0 3px 0;}
          .vl-status p {margin:0; color:#57606a;}
        </style>
        """))
        display(widgets.VBox([
            widgets.HTML("<div class='vl-hero'><h2>VisoLabel Colab Trainer</h2><p>Download, decrypt, validate, train, and upload checkpoints from one token.</p></div>"),
            widgets.VBox([self.token, self.api_url], layout=widgets.Layout(gap="8px")),
            widgets.HBox([self.run_button, self.resolve_button, self.train_button, self.upload_button, self.clear_button], layout=widgets.Layout(gap="8px", flex_flow="row wrap")),
            self.status,
            self.progress,
            self.summary,
            self.log_output,
        ], layout=widgets.Layout(gap="8px")))

    def _set_status(self, title: str, detail: str, kind: str):
        cls = f"vl-status {html.escape(kind)}".strip()
        self.status.value = f"<div class='{cls}'><h3>{html.escape(title)}</h3><p>{html.escape(detail)}</p></div>"
        self.progress.bar_style = {"ok": "success", "err": "danger", "active": "info"}.get(kind, "")

    def _log(self, message: str):
        ts = time.strftime("%H:%M:%S")
        with self.log_output:
            print(f"[{ts}] {message}", flush=True)

    def _render_summary(self):
        cfg = self.cfg or {}
        classes = cfg.get("classes") or []
        if not isinstance(classes, list):
            classes = []
        values = [
            ("Task", self.task or "-", self.command or "Waiting"),
            ("Model", str(cfg.get("model_name", "-")), f"{len(classes)} classes"),
            ("Epochs", str(cfg.get("epochs", "-")), f"Batch {cfg.get('batch_size', '-')}"),
            ("Image size", str(cfg.get("image_size", cfg.get("resolution", "-"))), "RF-DETR adjusts to a valid multiple if needed"),
            ("Dataset", f"{_count_images(self.dataset_dir) if self.dataset_dir else 0} images", str(self.dataset_dir or "-")),
        ]
        cards = []
        for label, value, sub in values:
            cards.append(f"<div class='vl-card'><div class='vl-label'>{html.escape(label)}</div><div class='vl-value'>{html.escape(value)}</div><div class='vl-sub'>{html.escape(sub)}</div></div>")
        self.summary.value = "<div class='vl-cards'>" + "".join(cards) + "</div>"

    def _post(self, endpoint: str, payload: dict) -> dict:
        url = _normalize_url(self.api_base, endpoint)
        response = self.session.post(url, json=payload, timeout=120)
        response.raise_for_status()
        return response.json()

    def _download(self, url: str, destination: Path, start: float, end: float):
        destination.parent.mkdir(parents=True, exist_ok=True)
        with self.session.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            total = int(response.headers.get("content-length") or 0)
            done = 0
            with open(destination, "wb") as f:
                iterator = response.iter_content(chunk_size=1024 * 1024)
                if total:
                    iterator = tqdm(iterator, total=max(math.ceil(total / (1024 * 1024)), 1), unit="MB", desc=destination.name)
                for chunk in iterator:
                    if not chunk:
                        continue
                    f.write(chunk)
                    done += len(chunk)
                    if total:
                        self.progress.value = start + min(done / total, 1.0) * (end - start)
        self._log(f"Downloaded {destination.name}: {_format_bytes(destination.stat().st_size)}")

    def _decrypt_bundle(self, encrypted_path: Path, output_path: Path, password: str, encryption: dict):
        if encryption.get("algorithm") != "AES-256-GCM":
            raise RuntimeError(f"Unsupported encryption algorithm: {encryption.get('algorithm')}")
        if encryption.get("kdf") != "PBKDF2-HMAC-SHA256":
            raise RuntimeError(f"Unsupported encryption KDF: {encryption.get('kdf')}")
        nonce = base64.b64decode(str(encryption["nonce_b64"]).encode("ascii"), validate=True)
        tag = base64.b64decode(str(encryption["tag_b64"]).encode("ascii"), validate=True)
        salt = base64.b64decode(str(encryption["salt_b64"]).encode("ascii"), validate=True)
        aad_value = encryption.get("aad_b64", "")
        aad = base64.b64decode(str(aad_value).encode("ascii"), validate=True) if aad_value else b"visolabel-colab-bundle-v1"
        iterations = int(encryption.get("kdf_iterations") or encryption.get("iterations") or 200000)
        kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=iterations)
        key = kdf.derive(password.encode("utf-8"))
        decryptor = Cipher(algorithms.AES(key), modes.GCM(nonce, tag)).decryptor()
        decryptor.authenticate_additional_data(aad)
        with open(encrypted_path, "rb") as src, open(output_path, "wb") as dst:
            for chunk in iter(lambda: src.read(1024 * 1024), b""):
                data = decryptor.update(chunk)
                if data:
                    dst.write(data)
            data = decryptor.finalize()
            if data:
                dst.write(data)

    def _extract_manifest(self):
        if BUNDLE_DIR.exists():
            shutil.rmtree(BUNDLE_DIR)
        BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
        _safe_extract_zip(BUNDLE_ZIP, BUNDLE_DIR)
        manifest_path = BUNDLE_DIR / "training_bundle.json"
        if not manifest_path.exists():
            raise RuntimeError("Bundle is missing training_bundle.json.")
        self.manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        self.task = self.manifest["task"]["task_type"]
        self.command = self.manifest["task"]["command"]
        self.cfg = self.manifest["training_config"]
        self.dataset_dir = BUNDLE_DIR / self.manifest["paths"]["dataset_dir"]
        if self.command == "train":
            reports = _validate_detection_dataset(self.dataset_dir)
        else:
            reports = _validate_classification_dataset(self.dataset_dir)
        (OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        (OUTPUT_DIR / "dataset_validation.json").write_text(json.dumps(reports, indent=2), encoding="utf-8")
        for report in reports:
            self._log(f"Validated {report['split']}: {report['images']} images, {report['annotations']} annotations, {report['categories']} categories, removed {report['removed_annotations']}.")
        self.train_button.disabled = False
        self._render_summary()

    def resolve_bundle(self):
        self.token_value = self.token.value.strip()
        self.api_base = self.api_url.value.strip() or DEFAULT_API_BASE_URL
        if not self.token_value:
            raise RuntimeError("Paste a VisoLabel Colab token first.")
        self.started_at = time.time()
        self.training_finished = False
        self.upload_button.disabled = True
        self.progress.value = 0
        WORK_DIR.mkdir(parents=True, exist_ok=True)
        for path in (BUNDLE_ENCRYPTED, BUNDLE_ZIP):
            if path.exists():
                path.unlink()
        self._set_status("Resolving bundle", "Downloading bundle metadata and preparing the dataset.", "active")
        self._log("Resolving token.")
        if self.token_value.startswith(TOKEN_PREFIX) or self.token_value.startswith(LEGACY_TOKEN_PREFIX):
            parsed = _parse_encrypted_token(self.token_value)
            self.api_token = parsed["server_token"]
            if parsed["api_url"] and self.api_url.value.strip() == DEFAULT_API_BASE_URL:
                self.api_base = parsed["api_url"]
                self.api_url.value = parsed["api_url"]
            bundle_url = parsed["bundle_url"]
            encryption = parsed["encryption"]
            password = parsed["password"]
            if not bundle_url or not encryption:
                payload = {"token": self.api_token}
                if parsed["bundle_id"]:
                    payload["bundle_id"] = parsed["bundle_id"]
                resolved = self._post("/api/v1/colab/resolve-token/", payload)
                bundle_url = bundle_url or _extract_bundle_url(resolved)
                encryption = encryption or _extract_encryption_metadata(resolved)
            if not bundle_url or not password or not encryption:
                raise RuntimeError("Encrypted token could not resolve bundle URL, password, or encryption metadata. Regenerate the token with the latest VisoLabel build.")
            self.progress.value = 5
            self._download(bundle_url, BUNDLE_ENCRYPTED, 5, 35)
            expected_cipher_sha = encryption.get("ciphertext_sha256", "")
            if expected_cipher_sha and _sha256(BUNDLE_ENCRYPTED) != expected_cipher_sha:
                raise RuntimeError("Encrypted bundle SHA-256 does not match token metadata.")
            self._set_status("Decrypting bundle", "The dataset is decrypted locally in this Colab runtime.", "active")
            self._log("Decrypting bundle locally.")
            self._decrypt_bundle(BUNDLE_ENCRYPTED, BUNDLE_ZIP, password, encryption)
            expected_plain_sha = encryption.get("plaintext_sha256", "")
            if expected_plain_sha and _sha256(BUNDLE_ZIP) != expected_plain_sha:
                raise RuntimeError("Decrypted bundle SHA-256 does not match token metadata.")
        else:
            self.api_token = self.token_value
            resolved = self._post("/api/v1/colab/resolve-token/", {"token": self.token_value})
            bundle_url = _extract_bundle_url(resolved)
            if not bundle_url:
                raise RuntimeError("bundle_url missing in resolve-token response.")
            self._download(bundle_url, BUNDLE_ZIP, 10, 40)
        self.progress.value = 42
        self._log("Extracting and validating bundle.")
        self._extract_manifest()
        self.progress.value = 50
        self._set_status("Bundle ready", "Review the preview, then start training.", "ok")

    def _install_training_deps(self):
        if self.command == "train":
            packages = ["rfdetr[train,loggers]", "pycocotools", "supervision"]
        else:
            packages = ["timm", "torchvision"]
        self._set_status("Installing dependencies", "Colab may take a few minutes the first time.", "active")
        self._log("Installing: " + " ".join(packages))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])

    def train(self):
        if not self.manifest:
            self.resolve_bundle()
        if OUTPUT_DIR.exists():
            shutil.rmtree(OUTPUT_DIR)
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        self._install_training_deps()
        RUNNER_PATH.write_text(RUNNER_CODE, encoding="utf-8")
        self._set_status("Training", "Live output is shown below. RF-DETR may spend several minutes downloading weights before the first epoch.", "active")
        self.progress.value = 55
        manifest_path = BUNDLE_DIR / "training_bundle.json"
        command = [sys.executable, "-u", str(RUNNER_PATH), str(manifest_path), str(self.dataset_dir), str(OUTPUT_DIR)]
        env = dict(os.environ)
        env["PYTHONUNBUFFERED"] = "1"
        with open(TRAINING_LOG, "w", encoding="utf-8") as log_file:
            def stream_training_text(text):
                if not text:
                    return
                log_file.write(text.replace("\r", "\n"))
                log_file.flush()
                with self.log_output:
                    print(text, end="", flush=True)
                if "Epoch" in text or "train/" in text or "%|" in text:
                    self.progress.value = min(max(self.progress.value, 60) + 0.2, 88)

            if os.name == "posix":
                import pty
                import select
                master_fd, slave_fd = pty.openpty()
                proc = subprocess.Popen(command, stdout=slave_fd, stderr=slave_fd, stdin=subprocess.DEVNULL, env=env, close_fds=True)
                os.close(slave_fd)
                try:
                    while True:
                        ready, _, _ = select.select([master_fd], [], [], 0.2)
                        if ready:
                            try:
                                data = os.read(master_fd, 4096)
                            except OSError:
                                break
                            if not data:
                                break
                            stream_training_text(data.decode("utf-8", errors="replace"))
                        elif proc.poll() is not None:
                            break
                finally:
                    os.close(master_fd)
                code = proc.wait()
            else:
                proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
                for raw in iter(proc.stdout.readline, ""):
                    stream_training_text(raw)
                code = proc.wait()
        if code != 0:
            raise RuntimeError(f"Training subprocess failed with exit code {code}. See {TRAINING_LOG}.")
        self.progress.value = 90
        self.training_finished = True
        self.upload_button.disabled = False
        self._set_status("Training finished", f"Outputs are in {OUTPUT_DIR}.", "ok")
        self._log(f"Training finished. Output dir: {OUTPUT_DIR}")

    def _request_upload(self, path: Path) -> dict:
        return self._post("/api/v1/colab/request-checkpoint-upload/", {
            "token": self.api_token,
            "file_name": path.name,
            "file_size": path.stat().st_size,
            "file_sha256": _sha256(path),
            "content_type": "application/octet-stream",
            "task_type": self.task,
        })

    def _upload_artifact(self, path: Path, index: int, total: int):
        info = self._request_upload(path)
        upload_url = str(info.get("upload_url") or info.get("url") or "")
        if not upload_url:
            raise RuntimeError(f"Upload URL missing for {path.name}.")
        method = str(info.get("method") or "PUT").upper()
        headers = dict(info.get("headers") or {})
        self._log(f"Uploading {path.name} ({_format_bytes(path.stat().st_size)}).")
        with open(path, "rb") as f:
            if method == "POST":
                response = requests.post(upload_url, data=f, headers=headers, timeout=600)
            else:
                response = requests.put(upload_url, data=f, headers=headers, timeout=600)
        response.raise_for_status()
        self.progress.value = 90 + (index / max(total, 1)) * 8

    def upload(self):
        if not self.api_token:
            raise RuntimeError("Resolve the bundle before uploading results.")
        self._set_status("Uploading results", "Sending checkpoints and logs back to VisoLabel.", "active")
        if TRAINING_LOG.exists():
            self.upload_list = [TRAINING_LOG]
        else:
            self.upload_list = []
        self.upload_list.extend(sorted(p for p in OUTPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in ARTIFACT_EXTENSIONS and p != TRAINING_LOG))
        seen = set()
        self.upload_list = [p for p in self.upload_list if not (str(p) in seen or seen.add(str(p)))]
        if not self.upload_list:
            self._log("No uploadable artifacts found.")
        for index, path in enumerate(self.upload_list, start=1):
            self._upload_artifact(path, index, len(self.upload_list))
        try:
            self._post("/api/v1/colab/finalize-run/", {"token": self.api_token})
        except Exception as exc:
            self._log(f"Finalize endpoint not available: {exc}")
        elapsed = int(time.time() - self.started_at) if self.started_at else 0
        self.progress.value = 100
        self._set_status("Complete", f"Uploaded {len(self.upload_list)} artifacts in {elapsed}s.", "ok")

    def _set_busy(self, busy: bool):
        self.run_button.disabled = busy
        self.resolve_button.disabled = busy
        self.train_button.disabled = busy or self.manifest is None
        self.upload_button.disabled = busy or not self.api_token or not self.training_finished

    def _run_action(self, action):
        self._set_busy(True)
        try:
            action()
        except Exception as exc:
            self._set_status("Error", str(exc), "err")
            self._log(f"ERROR: {exc}")
            raise
        finally:
            self._set_busy(False)

    def _on_resolve(self, _):
        self._run_action(self.resolve_bundle)

    def _on_train(self, _):
        self._run_action(self.train)

    def _on_upload(self, _):
        self._run_action(self.upload)

    def _on_run_full(self, _):
        def pipeline():
            self.resolve_bundle()
            self.train()
            self.upload()
        self._run_action(pipeline)


gui = VisoLabelColabGUI()
gui.display()
